# Point-in-time flat dataset — construction

Builds **one row per company** for a *forward-looking* propensity model:
*"which non-customers will take their first Lloyds charge in the next few years?"*

## How this differs from `data_flat.ipynb`

`data_flat.ipynb` measures features at `ASOF` and labels conversions in a window that **ends** at
`ASOF`. Most positives therefore converted *before* their features were measured, so the features
describe firms that had already been Lloyds customers for years. That is a **look-alike / profiling**
design: it answers *"who resembles our customers?"*, not *"who will become one?"*.

This notebook separates the two dates:

| | `data_flat.ipynb` | this notebook |
|---|---|---|
| features measured at | `ASOF` | `ASOF` |
| label window | `LABEL_SINCE` → `ASOF` (**before** features) | `ASOF` → `LABEL_END` (**after** features) |
| answers | who *resembles* a customer | who *will become* a customer |
| positives | more | fewer |
| supports a predictive claim | no | yes |

Everything else — the feature definitions, the SME filter, the sector mapping — is deliberately
identical, so the two tables are comparable and the *only* thing that changes is the direction of time.

## The one rule that makes this work

`first_lloyds` is computed from the **full** charge history, not the `ASOF`-cut history. The label has
to see forward; only the **features** are cut at `ASOF`. Getting this backwards (as the look-alike
build necessarily does) is what collapses the arrow of time.

Existing customers at `ASOF` are dropped: they are not prospects, and their conversion already
happened.

## 1 · Config — the knobs you manage

In [1]:
# --- portable paths: resolve the project root from ANY working directory ---
import sys
from pathlib import Path
_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
              if (p / "paths.py").is_file()), None)
if _ROOT is None:
    raise RuntimeError(
        "Cannot find paths.py in any parent of "
        f"{Path.cwd()} — run this notebook from inside the project folder.")
sys.path.insert(0, str(_ROOT))
from paths import COMPANIES_CSV, CHARGES_CSV, FLAT_POT_CSV

import numpy as np
import pandas as pd

# ============================ SETTINGS ============================
# ASOF = the snapshot date. EVERY feature is measured here and the charge history
# is cut here, so no feature can see the future.
ASOF = pd.Timestamp("2021-01-01")

# LABEL_END = the far edge of the prediction window. Label = 1 if the firm takes
# its FIRST Lloyds charge in (ASOF, LABEL_END].
#
# Window length is a real trade-off, measured on this data:
#   ASOF 2019 -> 7 yr window, ~1,965 positives (0.69%)  weaker claim, more signal
#   ASOF 2021 -> 5 yr window, ~1,315 positives (0.37%)  the balance point
#   ASOF 2023 -> 3 yr window, ~876   positives (0.20%)  sharper claim, thinner
# Keep LABEL_END <= the last date in charges_history.csv, or the most recent
# conversions are silently missing and every firm near the edge looks negative.
LABEL_END = pd.Timestamp("2026-01-01")

# Former customers who came back.
# A firm that borrowed from Lloyds years ago and has since repaid in full is
# arguably a prospect again: the relationship lapsed and its collateral is free.
# The strict rule excludes it anyway, because it once held a charge.
#   None -> strict. Exclude every firm holding a Lloyds charge by ASOF.
#   0.0  -> readmit a firm whose Lloyds charges were ALL satisfied by ASOF.
#   2.0  -> readmit only if the last was satisfied >= 2 years before ASOF.
# A firm with any Lloyds charge still OUTSTANDING at ASOF is never readmitted:
# it is a current customer, not a lapsed one.
# Measured at ASOF=2021: 1,196 firms took new Lloyds lending having borrowed
# before, but 1,080 still held an open charge. Only 116 are genuinely lapsed.
READMIT_YEARS = None

OUT_CSV = FLAT_POT_CSV      # API/flat_pot.csv — the table 4_model.ipynb reads
# ==================================================================
assert ASOF < LABEL_END, "ASOF must precede LABEL_END: the label window looks FORWARD"
print(f"features as-of {ASOF.date()}   label window ({ASOF.date()}, {LABEL_END.date()}]"
      f"   = {(LABEL_END - ASOF).days / 365.25:.1f} years")
print(f"output -> {OUT_CSV}")

features as-of 2021-01-01   label window (2021-01-01, 2026-01-01]   = 5.0 years
output -> /Users/natchalin_/Projects/final_project/Lloyds/API/flat_pot.csv


## 2 · Load the companies and their charge history

`charges_history.csv` is the label source (`is_lloyds`, set by the curated `LLOYDS_PATTERNS` regex in
Stage 3 of the CH pipeline). Non-Lloyds charges are safe as features — only Lloyds ones are the label.

**Note the asymmetry below.** `charges_asof` (cut at `ASOF`) feeds the *features*.
`first_lloyds` comes from the *uncut* history, because the label must be allowed to see forward.

In [2]:
# ONE company table now; is_sme flags the modelling population. Non-SME rows are
# kept in the file as the "already checked, do not re-pull" cache, so filter here.
companies = pd.read_csv(COMPANIES_CSV, dtype=str, low_memory=False)
companies = companies[companies["is_sme"].astype(str).str.lower().eq("true")].copy()
charges   = pd.read_csv(CHARGES_CSV, dtype=str)

companies["born"] = pd.to_datetime(companies["date_of_creation"], errors="coerce")
companies = companies.dropna(subset=["born"]).reset_index(drop=True)

charges["created_on"]   = pd.to_datetime(charges["created_on"], errors="coerce")
charges["satisfied_on"] = pd.to_datetime(charges["satisfied_on"], errors="coerce")
charges["is_lloyds"]    = charges["is_lloyds"].astype(str).str.lower().eq("true")
charges = charges.dropna(subset=["created_on"])

# THE POINT-IN-TIME CUT — features only. Without it, features read charges filed
# after ASOF and yrs_since_nonlloyds_chg goes NEGATIVE.
charges_asof = charges[charges["created_on"] <= ASOF]
nonlloyds    = charges_asof[~charges_asof["is_lloyds"]]

# THE LABEL — from the FULL history, deliberately. This is what makes the design
# forward-looking; cutting it at ASOF would make forward conversions invisible.
lloyds      = charges[charges["is_lloyds"]]
lloyds_pre  = lloyds[lloyds["created_on"] <= ASOF]     # already a customer by ASOF
lloyds_post = lloyds[lloyds["created_on"] >  ASOF]     # the label source

# `next_lloyds` (not `first_lloyds`) is the label: the first Lloyds charge taken
# AFTER the cutoff. For a never-customer that is also its first ever; for a
# readmitted former customer it is the return. One definition covers both.
companies["next_lloyds"] = companies["com_num"].map(
    lloyds_post.groupby("com_num")["created_on"].min())

# Was any pre-ASOF Lloyds charge still running at ASOF? Unsatisfied, or satisfied
# only after the cutoff, both count as open.
_open = lloyds_pre[lloyds_pre["satisfied_on"].isna() | (lloyds_pre["satisfied_on"] > ASOF)]
companies["held_lloyds_by_asof"] = companies["com_num"].isin(set(lloyds_pre["com_num"]))
companies["lloyds_open_at_asof"] = companies["com_num"].isin(set(_open["com_num"]))

# How long ago the last Lloyds facility was cleared — drives READMIT_YEARS.
_lastsat = lloyds_pre[lloyds_pre["satisfied_on"].notna()
                      & (lloyds_pre["satisfied_on"] <= ASOF)]
companies["yrs_since_lloyds_satisfied"] = (
    (ASOF - companies["com_num"].map(_lastsat.groupby("com_num")["satisfied_on"].max()))
    .dt.days / 365.25)

print(f"companies (SME)        : {len(companies):,}")
print(f"charge rows (all)      : {len(charges):,}")
print(f"charge rows <= ASOF    : {len(charges_asof):,}   "
      f"({len(charges) - len(charges_asof):,} held back from FEATURES as future)")
print(f"held a Lloyds charge by ASOF : {int(companies['held_lloyds_by_asof'].sum()):,}")
print(f"  still OPEN at ASOF         : {int(companies['lloyds_open_at_asof'].sum()):,}   <- current customers")
print(f"  all satisfied by ASOF      : {int((companies['held_lloyds_by_asof'] & ~companies['lloyds_open_at_asof']).sum()):,}   <- lapsed, READMIT_YEARS decides")
print(f"took a Lloyds charge after ASOF : {companies['next_lloyds'].notna().sum():,}   <- candidate positives")

companies (SME)        : 567,835
charge rows (all)      : 399,700
charge rows <= ASOF    : 257,505   (142,195 held back from FEATURES as future)
held a Lloyds charge by ASOF : 12,614
  still OPEN at ASOF         : 8,971   <- current customers
  all satisfied by ASOF      : 3,643   <- lapsed, READMIT_YEARS decides
took a Lloyds charge after ASOF : 3,201   <- candidate positives


## 3 · Eligibility and the label

Two conditions to be in the modelling population at `ASOF`:

1. **Existed** — incorporated on or before `ASOF`. A firm that did not yet exist could not have been
   scored, so including it would be a different kind of leakage.
2. **Not already a customer** — no Lloyds charge on or before `ASOF`. Existing customers are not
   prospects, and their conversion is in the past.

Then `label = 1` if the **next** Lloyds charge falls inside `(ASOF, LABEL_END]`. Note *next*, not
*first*: for a firm that never banked with Lloyds these are the same charge, but the distinction is
what lets a former customer be scored on its **return** rather than on borrowing it did years ago.

**`READMIT_YEARS` — former customers who came back.** The strict rule (`None`) excludes any firm that
ever held a Lloyds charge by `ASOF`. That is right for a firm still carrying one, but arguably wrong
for a firm that repaid in full years ago: the relationship has lapsed and its collateral is free, so
it is a prospect again. Setting `READMIT_YEARS = 0` brings those firms back into the population;
setting it to `2` demands the facility was cleared at least two years before the cutoff. A firm with
any Lloyds charge still **outstanding** at `ASOF` is never readmitted at any setting.

At `ASOF = 2021` this is a small population: of 1,196 firms that took new Lloyds lending having
borrowed before, 1,080 still held an open charge and only 116 had genuinely lapsed. The switch exists
so the choice is explicit and testable rather than silently baked in.

Firms whose first Lloyds charge falls **after** `LABEL_END` are labelled 0. That is correct: within
the window they did not convert. It also means the label is *right-censored* — worth one sentence in
the write-up.

In [3]:
d = companies.copy()
existed = d["born"] <= ASOF

# A lapsed former customer: held a Lloyds charge, none of it still open at ASOF,
# and the last one cleared at least READMIT_YEARS ago. NaN >= x is False, so
# firms that never borrowed from Lloyds can never be flagged lapsed.
if READMIT_YEARS is None:
    lapsed = pd.Series(False, index=d.index)
else:
    lapsed = (d["held_lloyds_by_asof"]
              & ~d["lloyds_open_at_asof"]
              & (d["yrs_since_lloyds_satisfied"] >= READMIT_YEARS))

d["readmitted"] = lapsed.astype(int)
d = d[existed & (~d["held_lloyds_by_asof"] | lapsed)].copy()

d["label"] = ((d["next_lloyds"] > ASOF) & (d["next_lloyds"] <= LABEL_END)).astype(int)

# how the conversions spread across the window -- informs whether it is too long
_hz = ((d.loc[d["label"] == 1, "next_lloyds"] - ASOF).dt.days / 365.25)

print(f"READMIT_YEARS = {READMIT_YEARS}")
print(f"rows       : {len(d):,}")
print(f"positives  : {int(d['label'].sum()):,}  ({d['label'].mean():.2%})")
print(f"readmitted : {int(d['readmitted'].sum()):,} former customers brought back in, "
      f"of which {int(d.loc[d['readmitted'] == 1, 'label'].sum()):,} convert in the window")
print(f"censored   : {int((d['next_lloyds'] > LABEL_END).sum()):,} firms convert after LABEL_END "
      f"-> labelled 0")
print(f"\nyears from ASOF to conversion (positives only):")
print(f"  median {_hz.median():.1f}   quartiles {_hz.quantile(.25):.1f} / {_hz.quantile(.75):.1f}")
print(_hz.round(0).value_counts().sort_index().rename("firms").to_string())

READMIT_YEARS = None
rows       : 360,603
positives  : 1,203  (0.33%)
readmitted : 0 former customers brought back in, of which 0 convert in the window
censored   : 208 firms convert after LABEL_END -> labelled 0

years from ASOF to conversion (positives only):
  median 1.8   quartiles 0.7 / 3.6
next_lloyds
0.0    241
1.0    287
2.0    178
3.0    178
4.0    195
5.0    124


## 4 · Features, measured as-of `ASOF`

Identical definitions to `data_flat.ipynb`, so the two tables can be compared directly.

All charge features come from `nonlloyds` (already cut at `ASOF`), so none of them can see the future.

`sector`, `region` and `account_type` are still **current snapshots** — Companies House does not serve
history for them cheaply. They are near-static, but this is the notebook's remaining leakage residual
and belongs in the write-up as a stated limitation.

`age_years` is safe here in a way it is not in the look-alike build: because the label window opens
*at* `ASOF` and every firm in the population existed by then, age no longer encodes *opportunity to
have converted already*. Section 6 still prints the diagnostic — check it rather than assuming.

In [4]:
g = nonlloyds.groupby("com_num")

d["nonlloyds_charges"]    = d["com_num"].map(g.size()).fillna(0).astype(int)
d["has_nonlloyds_charge"] = (d["nonlloyds_charges"] > 0).astype(int)
d["n_distinct_lenders"]   = d["com_num"].map(g["persons_entitled"].nunique()).fillna(0).astype(int)

# 50 = "never borrowed" sentinel; clip keeps the never-borrowed and long-ago cases together
_last = d["com_num"].map(g["created_on"].max())
d["yrs_since_nonlloyds_chg"] = ((ASOF - _last).dt.days / 365.25).fillna(50).clip(0, 50)

# ---- charge DYNAMICS: not WHETHER a firm borrowed, but WHEN and in which direction ----
# 1) momentum -- borrowing in the recent past, not over the whole company lifetime
_recent = nonlloyds[nonlloyds["created_on"] >= ASOF - pd.DateOffset(months=24)]
d["n_charges_last_24m"] = d["com_num"].map(_recent.groupby("com_num").size()).fillna(0).astype(int)

# 2) leverage direction -- 3 outstanding charges (encumbered, maybe at capacity) is a
#    very different lead from 3 settled ones (proven borrower, collateral freed up)
_sat = nonlloyds[nonlloyds["satisfied_on"].notna() & (nonlloyds["satisfied_on"] <= ASOF)]
d["n_satisfied"]   = d["com_num"].map(_sat.groupby("com_num").size()).fillna(0).astype(int)
d["n_outstanding"] = (d["nonlloyds_charges"] - d["n_satisfied"]).clip(lower=0)

# 3) the re-lend trigger -- a firm that has just cleared a loan has freed collateral
#    and a demonstrated repayment record. 50 = "never settled one".
_lastsat = d["com_num"].map(_sat.groupby("com_num")["satisfied_on"].max())
d["yrs_since_satisfaction"] = ((ASOF - _lastsat).dt.days / 365.25).fillna(50).clip(0, 50)

d["age_years"] = (ASOF - d["born"]).dt.days / 365.25
d["accounts_overdue"] = d["accounts_overdue"].map(
    {"True": 1, True: 1, "False": 0, False: 0}).fillna(0).astype(int)

SEC = [(1,3,"A: Agriculture"),(5,9,"B: Mining"),(10,33,"C: Manufacturing"),(35,35,"D: Utilities"),
       (36,39,"E: Water/Waste"),(41,43,"F: Construction"),(45,47,"G: Retail/Wholesale"),
       (49,53,"H: Transport"),(55,56,"I: Accommodation/Food"),(58,63,"J: Information/Comms"),
       (64,66,"K: Finance/Insurance"),(68,68,"L: Real Estate"),(69,75,"M: Professional/Scientific"),
       (77,82,"N: Admin Support"),(84,84,"O: Public Admin"),(85,85,"P: Education"),
       (86,88,"Q: Health/Social"),(90,93,"R: Arts/Recreation"),(94,96,"S: Other Services"),
       (97,98,"T: Household Activities"),(99,99,"U: Extraterritorial")]

def section(code):
    """SIC code -> Companies House SIC section. 98 (property management) -> Real Estate."""
    try:
        div = int(str(code).strip()[:2])
    except (ValueError, TypeError):
        return None
    if div == 98:
        return "L: Real Estate"
    for lo, hi, s in SEC:
        if lo <= div <= hi:
            return s
    return None

d["sector"] = d["sic_code"].map(section)
print(f"sector mapped: {d['sector'].notna().mean():.1%}")

sector mapped: 100.0%


## 5 · Save

In [5]:
KEEP = ["com_num", "name", "sector", "account_type", "accounts_overdue",
        "nonlloyds_charges", "has_nonlloyds_charge", "yrs_since_nonlloyds_chg",
        "n_distinct_lenders",
        "n_charges_last_24m", "n_outstanding", "n_satisfied", "yrs_since_satisfaction",
        "age_years", "readmitted", "label"]

if "region" in d.columns and d["region"].notna().mean() > 0.5:
    # firms with no post_code have no derivable region -> label them explicitly rather
    # than leaving NaN, which would break sklearn downstream.
    d["region"] = d["region"].fillna("unknown")
    KEEP.insert(3, "region")
else:
    print("NOTE: 'region' missing/sparse -> excluded. Run Part 5 of GDELT.ipynb to add it.")

flat = d[KEEP].dropna(subset=["sector"]).reset_index(drop=True)
flat.to_csv(OUT_CSV, index=False)

print(f"saved {len(flat):,} rows | positives {int(flat['label'].sum()):,} "
      f"({flat['label'].mean():.2%})  ->  {OUT_CSV}")
flat.head(3)

saved 360,600 rows | positives 1,203 (0.33%)  ->  /Users/natchalin_/Projects/final_project/Lloyds/API/flat_pot.csv


,com_num,name,sector,region,account_type,accounts_overdue,nonlloyds_charges,has_nonlloyds_charge,yrs_since_nonlloyds_chg,n_distinct_lenders,n_charges_last_24m,n_outstanding,n_satisfied,yrs_since_satisfaction,age_years,readmitted,label
0,07852962,369 UPLAND ROAD RTM COMPANY LIMITED,L: Real Estate,London,micro-entity,0,0,0,50.0,0,0,0,0,50.0,9.122519,0,0
1,12047798,36A TACHBROOK STREET LIMITED,S: Other Services,London,micro-entity,0,0,0,50.0,0,0,0,0,50.0,1.557837,0,0
2,10379693,36ALPHA LTD,J: Information/Comms,London,micro-entity,0,0,0,50.0,0,0,0,0,50.0,4.292950,0,0


## 6 · Sanity checks

The first block is the one that matters. Three assertions define this design:

1. **No negative `yrs_since_nonlloyds_chg`** — a negative value means a feature saw a charge filed
   after `ASOF`.
2. **No positive converted on or before `ASOF`** — if one did, the existing-customer exclusion failed
   and the label is looking backwards.
3. **No firm in the table held a Lloyds charge at `ASOF`** — the population is prospects only.

The age table is the exposure diagnostic. In the look-alike build a **rising monotonic** pattern means
you are measuring how many years a firm has had to convert, not its propensity. Here the window opens
at `ASOF` for everyone, so that artefact should be gone — if age still rises monotonically, it is
telling you something real about older firms, but check it before trusting `age_years` in the model.

In [6]:
neg = int((flat["yrs_since_nonlloyds_chg"] < 0).sum())
print(f"negative yrs_since_nonlloyds_chg : {neg}   <- must be 0")
print(f"rows with any NaN                : {int(flat.isna().any(axis=1).sum())}")

_pos = d[d["label"] == 1]
assert neg == 0, "a feature saw a charge filed after ASOF"
assert (_pos["next_lloyds"] > ASOF).all(), "a positive converted on/before ASOF"
assert not d["lloyds_open_at_asof"].any(), "a current Lloyds customer survived the exclusion"
assert (d.loc[d["readmitted"] == 1, "held_lloyds_by_asof"]).all(), \
    "a firm was readmitted without ever having been a customer"
print("all four point-in-time assertions passed")

if d["readmitted"].any():
    _r = d.groupby("readmitted")["label"].agg(["size", "mean"])
    _r.columns = ["firms", "conversion_rate"]
    print("\nnever-customer vs readmitted former customer:")
    print(_r.assign(conversion_rate=(_r["conversion_rate"] * 100).round(2)).to_string())

print("\nmean by group (1 = converted in window, 0 = did not):")
num = ["age_years", "nonlloyds_charges", "has_nonlloyds_charge",
       "yrs_since_nonlloyds_chg", "n_distinct_lenders",
       "n_charges_last_24m", "n_outstanding", "n_satisfied", "yrs_since_satisfaction"]
print(flat.groupby("label")[num].mean().round(2).to_string())

print("\nconversion rate by age band  (rising monotonic => check for exposure artefact):")
bands = pd.cut(flat["age_years"], [0, 5, 10, 15, 20, 30, 200],
               labels=["0-5", "5-10", "10-15", "15-20", "20-30", "30+"])
tab = flat.groupby(bands, observed=True)["label"].agg(["size", "mean"])
tab.columns = ["firms", "label_rate"]
print(tab.assign(label_rate=(tab["label_rate"] * 100).round(2)).to_string())

# the charge features overlap by construction -- check before putting them all in a
# model. |r| > 0.8 between two of them means keep one, not both.
print("\ncorrelation among the charge features:")
chg = ["nonlloyds_charges", "has_nonlloyds_charge", "yrs_since_nonlloyds_chg",
       "n_distinct_lenders", "n_charges_last_24m", "n_outstanding", "n_satisfied",
       "yrs_since_satisfaction"]
print(flat[chg].corr().round(2).to_string())

print("\ntop sectors by label rate (min 500 firms):")
s = flat.groupby("sector")["label"].agg(["size", "mean"])
s = s[s["size"] >= 500].sort_values("mean", ascending=False)
s.columns = ["firms", "label_rate"]
print(s.assign(label_rate=(s["label_rate"] * 100).round(2)).head(8).to_string())

negative yrs_since_nonlloyds_chg : 0   <- must be 0
rows with any NaN                : 0
all four point-in-time assertions passed

mean by group (1 = converted in window, 0 = did not):
       age_years  nonlloyds_charges  has_nonlloyds_charge  yrs_since_nonlloyds_chg  n_distinct_lenders  n_charges_last_24m  n_outstanding  n_satisfied  yrs_since_satisfaction
label                                                                                                                                                                         
0           9.59               0.50                  0.15                    43.59                0.27                0.08           0.33         0.18                   47.79
1           9.68               1.31                  0.35                    34.46                0.61                0.20           0.79         0.52                   44.60

conversion rate by age band  (rising monotonic => check for exposure artefact):
            firms  label_rate
age_

## 7 · Next — the lead list

`flat_pot.csv` is the **training** table. Unlike the look-alike build it contains **no existing
customers at all** (they were excluded at `ASOF`), so every row is a genuine prospect and the lead
list is simply the top of the ranked scores — no `label == 1` drop needed.

Because each firm appears exactly once, there is no repeated-firm correlation, so no cluster bootstrap
or `GroupKFold` is required.

**Judge it on lift, not raw Precision@K.** Prevalence here (~0.4%) is far below the look-alike table's
(~1.8%), so raw precision is mechanically lower while the underlying signal is comparable. Reporting
lift makes the two designs comparable; reporting raw precision alone makes this one look worse than it
is.

**For deployment**, re-run this notebook with `ASOF = today` and no label (every conversion is still in
the future), score, and rank. The feature definitions must match exactly — the definitions *are* the
model.